# ODS 层 — 原始数据入仓

将黄色 + 绿色出租车原始 Parquet 写入 ODS 层，保留原始字段，追加 `source_type` 和 `ingestion_ts` 元数据列，按 `source_type / pickup_year / pickup_month` 分区存储。

## 1. 初始化 SparkSession

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TLC-ODS-Ingest")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE      = os.path.expanduser("~/tlc-spark-pipeline")
RAW_Y     = f"{BASE}/data/raw/yellow"
RAW_G     = f"{BASE}/data/raw/green"
RAW_FHV   = f"{BASE}/data/raw/fhv_hv"
ODS_Y     = f"{BASE}/data/warehouse/ods/yellow"
ODS_G     = f"{BASE}/data/warehouse/ods/green"
ODS_FHV   = f"{BASE}/data/warehouse/ods/fhv_hv"

TARGET_MONTH = os.environ.get("TARGET_MONTH", "2024-03")
yr, mo = int(TARGET_MONTH.split("-")[0]), int(TARGET_MONTH.split("-")[1])
print(f"TARGET_MONTH={TARGET_MONTH}")

print(f"Spark {spark.version} ready")


26/10/19 14:43:02 WARN Utils: Your hostname, jelly-macbook.local resolves to a loopback address: 127.0.0.1; using 192.168.1.229 instead (on interface en0)
26/10/19 14:43:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/19 14:43:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


TARGET_MONTH=2024-03
Spark 3.5.1 ready


## 2. 黄色出租车 → ODS

In [2]:
%%time
raw_yellow = spark.read.parquet(f"{RAW_Y}/*{TARGET_MONTH}*.parquet")

ods_yellow = (
    raw_yellow
    .withColumn("source_type",   F.lit("yellow"))
    .withColumn("ingestion_ts",  F.current_timestamp())
    # 统一时间字段名（为 DWD 统一做准备）
    .withColumnRenamed("tpep_pickup_datetime",  "pickup_datetime")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
    # 分区列
    .withColumn("pickup_year",  F.year("pickup_datetime"))
    .withColumn("pickup_month", F.month("pickup_datetime"))
)

print(f"Yellow raw rows: {raw_yellow.count():,}")
ods_yellow.printSchema()

Yellow raw rows: 3,582,628
root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- source_type: string (nullable = false)
 |-- ingestion_ts: timestamp (nullable = false)
 |-- pickup_year:

In [3]:
%%time
(
    ods_yellow
    .write
    .partitionBy("pickup_year", "pickup_month")
    .mode("overwrite")
    .parquet(ODS_Y)
)
print(f"Yellow ODS written to: {ODS_Y}")

Yellow ODS written to: /Users/lujiayi/tlc-spark-pipeline/data/warehouse/ods/yellow
CPU times: user 7.17 ms, sys: 5.9 ms, total: 13.1 ms
Wall time: 7.98 s


## 3. 绿色出租车 → ODS

In [4]:
%%time
raw_green = spark.read.parquet(f"{RAW_G}/*{TARGET_MONTH}*.parquet")

ods_green = (
    raw_green
    .withColumn("source_type",   F.lit("green"))
    .withColumn("ingestion_ts",  F.current_timestamp())
    # 统一时间字段名
    .withColumnRenamed("lpep_pickup_datetime",  "pickup_datetime")
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
    .withColumn("pickup_year",  F.year("pickup_datetime"))
    .withColumn("pickup_month", F.month("pickup_datetime"))
)

print(f"Green raw rows: {raw_green.count():,}")
ods_green.printSchema()

Green raw rows: 57,457
root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- source_type: string (nullable = false)
 |-- ingestion_ts: timestamp (nu

In [5]:
%%time
(
    ods_green
    .write
    .partitionBy("pickup_year", "pickup_month")
    .mode("overwrite")
    .parquet(ODS_G)
)
print(f"Green ODS written to: {ODS_G}")

Green ODS written to: /Users/lujiayi/tlc-spark-pipeline/data/warehouse/ods/green
CPU times: user 3.33 ms, sys: 3.08 ms, total: 6.4 ms
Wall time: 980 ms


## 4. FHV HV 网约车 → ODS

Uber/Lyft 等网约车，列名已为 `pickup_datetime`，补充元数据列即可。

In [6]:
%%time
raw_fhv = spark.read.parquet(f"{RAW_FHV}/*{TARGET_MONTH}*.parquet")

ods_fhv = (
    raw_fhv
    .withColumn("source_type",  F.lit("fhv_hv"))
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("pickup_year",  F.year("pickup_datetime"))
    .withColumn("pickup_month", F.month("pickup_datetime"))
)

print(f"FHV HV raw rows: {raw_fhv.count():,}")
ods_fhv.printSchema()


FHV HV raw rows: 21,280,788
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nul

In [7]:
%%time
(
    ods_fhv
    .write
    .partitionBy("pickup_year", "pickup_month")
    .mode("overwrite")
    .parquet(ODS_FHV)
)
print(f"FHV HV ODS written to: {ODS_FHV}")


26/10/19 14:43:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


26/10/19 14:43:20 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


FHV HV ODS written to: /Users/lujiayi/tlc-spark-pipeline/data/warehouse/ods/fhv_hv
CPU times: user 24.7 ms, sys: 31.4 ms, total: 56.1 ms
Wall time: 39.4 s


## 4. 验证 ODS

In [8]:
for source, path in [("yellow", ODS_Y), ("green", ODS_G), ("fhv_hv", ODS_FHV)]:
    df = spark.read.parquet(path)
    cnt = df.count()
    print(f"{source}: {cnt:,} rows")
    df.groupBy("pickup_year", "pickup_month").count().orderBy("pickup_year", "pickup_month").show()

spark.stop()
print("ODS ingest complete.")


yellow: 3,582,628 rows


+-----------+------------+-------+
|pickup_year|pickup_month|  count|
+-----------+------------+-------+
|       2002|          12|      2|
|       2024|           2|     19|
|       2024|           3|3582605|
|       2024|           4|      2|
+-----------+------------+-------+



green: 57,457 rows


+-----------+------------+-----+
|pickup_year|pickup_month|count|
+-----------+------------+-----+
|       2008|          12|    2|
|       2024|           2|    7|
|       2024|           3|57447|
|       2024|           4|    1|
+-----------+------------+-----+



fhv_hv: 21,280,788 rows


+-----------+------------+--------+
|pickup_year|pickup_month|   count|
+-----------+------------+--------+
|       2024|           3|21280788|
+-----------+------------+--------+



ODS ingest complete.
